# Refactoring Code & Unit Testing

### Loading Libraries

In [ ]:
# Numerical Computing
import math
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualisation
import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt

# SciPy
from scipy.ndimage import shift

# YahooFinance
import yfinance as yf

# PyTest
import pytest

In [ ]:
# port_tickers = [
#     'QCOM', 'TSLA', 'NFLX', 'DIS', 'PG',
#     'MMM', 'IBM', 'BRK-B', 'UPS', 'F'
# ]

# bm_ticker = '^GSPC'

# ticker_list = [bm_ticker] + port_tickers

# period = '6mo'

# price_df_rf = get_tickers(
#     ticker_list=ticker_list,
#     period=period
# )

# price_df_rf.to_csv('data/tickers-raw.csv')

In [ ]:
%%writefile test_pd_refactor.py

@pytest.fixture
def raw_price_df():
    return pd.read_csv(
        'data/tickers-raw.csv',
        index_col='Date',
        parse_dates=['Date'],
        dtype_backend='pyarrow',
        engine='pyarrow'
    )


def test_basic(raw_price_df):
    assert len(raw_price_df) > 1

In [ ]:
%%writefile returns.py

def np_returns(df, col):
    values = df[col].to_numpy()

    shifted = shift(values, 1, cval=np.NaN)

    res = np.round(
        np.subtract(
            np.exp(
                np.nancumsum(
                    np.log(
                        np.divide(values, shifted)
                    )
                )
            ),
            1
        ),
        3
    )

    res[0] = np.NaN

    return pd.Series(res, index=df.index)